## Feature Engineering

In [9]:
# Import the libraries
import pandas as pd

# Load clean combined data from Notebook 01
df = pd.read_csv("../data/processed/all_indices_clean.csv")
df["Date"] = pd.to_datetime(df["Date"])

df.info()
print(df.head())

<class 'pandas.DataFrame'>
RangeIndex: 5904 entries, 0 to 5903
Data columns (total 21 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Date          5904 non-null   datetime64[us]
 1   High_(t-1)    5904 non-null   float64       
 2   Low_(t-1)     5904 non-null   float64       
 3   Close_(t-1)   5904 non-null   float64       
 4   Open_(t-1)    5904 non-null   float64       
 5   Volume_(t-1)  5904 non-null   int64         
 6   SMA_6         5904 non-null   float64       
 7   EMA_26        5904 non-null   float64       
 8   EMA_12        5904 non-null   float64       
 9   MACD          5904 non-null   float64       
 10  MACD_Signal   5904 non-null   float64       
 11  RSI_6         5904 non-null   float64       
 12  RSV_9         5904 non-null   float64       
 13  PSY_13        5904 non-null   float64       
 14  K_9           5904 non-null   float64       
 15  D_3           5904 non-null   float64       
 16 

In [2]:
# Feature 1: yesterday's daily return % (momentum) - per index
df["Prev_Return_%"] = df.groupby("Index")["Close_(t-1)"].pct_change(fill_method=None) * 100

# Feature 2: yesterday's High-Low range as % of its close (how wild the day was)
df["Prev_Range_%"] = (df["High_(t-1)"] - df["Low_(t-1)"]) / df["Close_(t-1)"] * 100

print(df[["Close_(t-1)", "Prev_Return_%", "Prev_Range_%"]].head())

   Close_(t-1)  Prev_Return_%  Prev_Range_%
0      6074.80            NaN      0.444459
1      6049.90      -0.409890      1.146300
2      6055.75       0.096696      0.459068
3      6034.75      -0.346778      0.545176
4      5998.90      -0.594059      1.162713


In [3]:
# Feature 3: 30-day rolling volatility of PAST returns (risk regime we found in EDA)
df["Volatility_30d"] = df.groupby("Index")["Prev_Return_%"].transform(lambda x: x.rolling(30).std())

# Feature 4: how far yesterday's close sits above/below its 6-day average (%)
df["MA_Distance_%"] = (df["Close_(t-1)"] - df["SMA_6"]) / df["SMA_6"] * 100

print(df[["Prev_Return_%", "Volatility_30d", "MA_Distance_%"]].head(35))

    Prev_Return_%  Volatility_30d  MA_Distance_%
0             NaN             NaN       0.261045
1       -0.409890             NaN      -0.060846
2        0.096696             NaN       0.015828
3       -0.346778             NaN      -0.277340
4       -0.594059             NaN      -0.813889
5       -0.194202             NaN      -0.767513
6       -0.506911             NaN      -0.947942
7        0.038611             NaN      -0.659994
8       -0.342328             NaN      -0.677337
9       -0.594396             NaN      -0.905173
10      -0.095706             NaN      -0.719346
11       0.417949             NaN      -0.122967
12       0.176446             NaN       0.120659
13      -0.606781             NaN      -0.312319
14      -0.161948             NaN      -0.329419
15       0.183443             NaN      -0.131646
16       0.703604             NaN       0.452397
17       0.056400             NaN       0.450867
18      -1.527835             NaN      -0.858473
19      -0.033321   

In [5]:
# Feature 5: MACD histogram = MACD minus its signal line (strength of momentum)
df["MACD_Hist"] = df["MACD"] - df["MACD_Signal"]

# Feature 6: day of the week (0 = Monday) - calendar behaviour
df["DayOfWeek"] = df["Date"].dt.dayofweek

print(df[["MACD", "MACD_Signal", "MACD_Hist", "Date", "DayOfWeek"]].head())

        MACD  MACD_Signal  MACD_Hist       Date  DayOfWeek
0  26.516723    22.432756   4.083967 2013-01-28          0
1  25.211598    22.988524   2.223074 2013-01-29          1
2  24.368420    23.264503   1.103916 2013-01-30          2
3  21.754894    22.962581  -1.207688 2013-01-31          3
4  16.599511    21.689967  -5.090456 2013-02-01          4


C:\Users\DELL\AppData\Local\Temp\ipykernel_11012\36376371.py:5: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["DayOfWeek"] = df["Date"].dt.dayofweek


In [6]:
# Target for our direction models: 1 = today closed UP, 0 = DOWN
df["Direction"] = (df["Close_(t)"] > df["Close_(t-1)"]).astype(int)

print(df["Direction"].value_counts())
print("UP days share:", round(df["Direction"].mean() * 100, 1), "%")

Direction
1    3304
0    2600
Name: count, dtype: int64
UP days share: 56.0 %


In [7]:
# groupby pct_change and rolling() again create NaN in the first rows - drop, like Notebook 01
before = len(df)
df = df.dropna().reset_index(drop=True)
print("Dropped", before - len(df), "new warm-up rows ->", len(df), "final rows")
print("Missing values left:", df.isnull().sum().sum())

Dropped 0 new warm-up rows -> 5904 final rows
Missing values left: 0


In [8]:
# Save the final model-ready dataset
df.to_csv("../data/processed/model_ready.csv", index=False)
print("Saved ../data/processed/model_ready.csv with shape:", df.shape)
df.info()

Saved ../data/processed/model_ready.csv with shape: (5904, 24)
<class 'pandas.DataFrame'>
RangeIndex: 5904 entries, 0 to 5903
Data columns (total 24 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Date          5904 non-null   datetime64[us]
 1   High_(t-1)    5904 non-null   float64       
 2   Low_(t-1)     5904 non-null   float64       
 3   Close_(t-1)   5904 non-null   float64       
 4   Open_(t-1)    5904 non-null   float64       
 5   Volume_(t-1)  5904 non-null   int64         
 6   SMA_6         5904 non-null   float64       
 7   EMA_26        5904 non-null   float64       
 8   EMA_12        5904 non-null   float64       
 9   MACD          5904 non-null   float64       
 10  MACD_Signal   5904 non-null   float64       
 11  RSI_6         5904 non-null   float64       
 12  RSV_9         5904 non-null   float64       
 13  PSY_13        5904 non-null   float64       
 14  K_9           5904 non-null   float6

### Conclusion - Feature Engineering
1. Built 6 new input features: Prev_Return_%, Prev_Range_%, Volatility_30d, MA_Distance_%, MACD_Hist, DayOfWeek.
2. Built the classification target Direction (UP=1 / DOWN=0); UP days = 56%.
3. Every feature uses only BEFORE-today information - no data leakage.
4. Dropped 60 new warm-up rows created by rolling/pct_change -> 5,844 rows, 0 missing.
5. Saved model_ready.csv (5,844 rows x 28 columns).
